In [13]:
import pandas as pd

In [14]:
# train data consist of data cycle until failure
train_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/raw/train_FD003.txt", sep=r"\s+", header=None)
# test data stop before failure to predict the rul
test_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/raw/test_FD003.txt", sep=r"\s+", header=None)
rul_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/raw/RUL_FD003.txt", sep=r"\s+", header=None)

In [15]:
train_df.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,1,1,-0.0005,0.0004,100.0,518.67,642.36,1583.23,1396.84,14.62,...,522.31,2388.01,8145.32,8.4246,0.03,391,2388,100.0,39.11,23.3537
1,1,2,0.0008,-0.0003,100.0,518.67,642.50,1584.69,1396.89,14.62,...,522.42,2388.03,8152.85,8.4403,0.03,392,2388,100.0,38.99,23.4491
2,1,3,-0.0014,-0.0002,100.0,518.67,642.18,1582.35,1405.61,14.62,...,522.03,2388.00,8150.17,8.3901,0.03,391,2388,100.0,38.85,23.3669
3,1,4,-0.0020,0.0001,100.0,518.67,642.92,1585.61,1392.27,14.62,...,522.49,2388.08,8146.56,8.3878,0.03,392,2388,100.0,38.96,23.2951
4,1,5,0.0016,0.0000,100.0,518.67,641.68,1588.63,1397.65,14.62,...,522.58,2388.03,8147.80,8.3869,0.03,392,2388,100.0,39.14,23.4583


In [16]:
test_df.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,1,1,-0.0017,-0.0004,100.0,518.67,641.94,1581.93,1396.93,14.62,...,521.89,2387.94,8133.48,8.3760,0.03,391,2388,100.0,39.07,23.4468
1,1,2,0.0006,-0.0002,100.0,518.67,642.02,1584.86,1398.90,14.62,...,521.85,2388.01,8137.44,8.4062,0.03,391,2388,100.0,39.04,23.4807
2,1,3,0.0014,-0.0003,100.0,518.67,641.68,1581.78,1391.92,14.62,...,522.10,2387.94,8138.25,8.3553,0.03,391,2388,100.0,39.10,23.4244
3,1,4,0.0027,0.0001,100.0,518.67,642.20,1584.53,1395.34,14.62,...,522.45,2387.96,8137.07,8.3709,0.03,392,2388,100.0,38.97,23.4782
4,1,5,-0.0001,0.0001,100.0,518.67,642.46,1589.03,1395.86,14.62,...,521.91,2387.97,8134.20,8.4146,0.03,391,2388,100.0,39.09,23.3950


In [17]:
rul_df.head()

,0
0,44
1,51
2,27
3,120
4,101


In [18]:
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print("rul_df shape:", rul_df.shape)

train_df shape: (24720, 26)
test_df shape: (16596, 26)
rul_df shape: (100, 1)


In [19]:
# prepare name for each columns
cols_name = (
    ['engine_id','cycle'] +
    [f'setting_{i}' for i in range(1,4)] +
    [f'sensor_{i}' for i in range(1,22)]
)


def preprocess(train_data, test_data, rul_data, cols, clip=False):
    # load and rename columns
    train = train_data
    train.columns = cols

    test = test_data
    test.columns = cols

    rul = rul_data
    rul.columns = ["rul"]

    # compute RUL
    max_cycle = train.groupby("engine_id")["cycle"].max()

    train["rul"] = train.apply(
        lambda row: max_cycle[row.engine_id] - row.cycle,
        axis=1
    )

    # optional clip to improves training
    if clip == True:
        train["rul"] = train["rul"].clip(upper=125)

    return (train, test, rul)

In [20]:
train_df, test_df, rul_df = preprocess(train_df, test_df, rul_df, cols_name, clip=True)

In [21]:
train_df.head()

,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,rul
0,1,1,-0.0005,0.0004,100.0,518.67,642.36,1583.23,1396.84,14.62,...,2388.01,8145.32,8.4246,0.03,391,2388,100.0,39.11,23.3537,125.0
1,1,2,0.0008,-0.0003,100.0,518.67,642.50,1584.69,1396.89,14.62,...,2388.03,8152.85,8.4403,0.03,392,2388,100.0,38.99,23.4491,125.0
2,1,3,-0.0014,-0.0002,100.0,518.67,642.18,1582.35,1405.61,14.62,...,2388.00,8150.17,8.3901,0.03,391,2388,100.0,38.85,23.3669,125.0
3,1,4,-0.0020,0.0001,100.0,518.67,642.92,1585.61,1392.27,14.62,...,2388.08,8146.56,8.3878,0.03,392,2388,100.0,38.96,23.2951,125.0
4,1,5,0.0016,0.0000,100.0,518.67,641.68,1588.63,1397.65,14.62,...,2388.03,8147.80,8.3869,0.03,392,2388,100.0,39.14,23.4583,125.0


In [22]:
test_df.head()

,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0017,-0.0004,100.0,518.67,641.94,1581.93,1396.93,14.62,...,521.89,2387.94,8133.48,8.3760,0.03,391,2388,100.0,39.07,23.4468
1,1,2,0.0006,-0.0002,100.0,518.67,642.02,1584.86,1398.90,14.62,...,521.85,2388.01,8137.44,8.4062,0.03,391,2388,100.0,39.04,23.4807
2,1,3,0.0014,-0.0003,100.0,518.67,641.68,1581.78,1391.92,14.62,...,522.10,2387.94,8138.25,8.3553,0.03,391,2388,100.0,39.10,23.4244
3,1,4,0.0027,0.0001,100.0,518.67,642.20,1584.53,1395.34,14.62,...,522.45,2387.96,8137.07,8.3709,0.03,392,2388,100.0,38.97,23.4782
4,1,5,-0.0001,0.0001,100.0,518.67,642.46,1589.03,1395.86,14.62,...,521.91,2387.97,8134.20,8.4146,0.03,391,2388,100.0,39.09,23.3950


In [23]:
rul_df.head()

,rul
0,44
1,51
2,27
3,120
4,101


In [24]:
# save preprocessed data
train_df.to_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/raw/train2.csv", index=False)
test_df.to_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/raw/test2.csv", index=False)
rul_df.to_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/raw/rul2.csv", index=False)